# HydroShare S3 Data Access

**Purpose**: This notebook demonstrates best practices for integrating HydroShare data into research workflows by leveraging it's S3 data storage capability.

In [ ]:
import hsclient
import getpass
import requests

## 1. Connecting to HydroShare Files via S3

This section outlines techniques for working with HydroShare data using Python libraries.

In [ ]:
# Sign in to HydroShare using the hsclient library
# This will prompt you to enter your username and password

username = input("Username: ").strip()
password = getpass.getpass("Password for {}: ".format(username))
hs = hsclient.HydroShare(username, password)

Load a HydroShare resource that we're intersted in: https://hydroshare.org/resource/5b379c908019453e86f304e0a07c4984/ 

In [ ]:
resource_id = '5b379c908019453e86f304e0a07c4984'
resource = hs.resource(resource_id)

Now that the resource has been loaded, we can use `hsclient` functions to browse the files. The `hsclient` library can be use to create, modify, and interact with HydroShare resources. It was designed to allow you to write code to do pretty much everything you can do through HydroShare's web user interface. documentation on of all its capabilities can be found at: https://github.com/hydroshare/hsclient.

Let's explore some of the most basic capabilites, such as listing files and exploring their metadata.

In [ ]:
# Print some basic metadata
print(f'Title: {resource.metadata.title}')
print(f'Description: {resource.metadata.abstract}')

In [ ]:
# List the contents of the HydroShare resource
resource.files()

This library can also be used to upload and download files. This is left as an exercise to the reader.

## 2. Interacting with HydroShare Files via Python S3FS

This section demonstrates how to interact with HydroShare content using the S3FS Python library. This is particularly useful when working with large data in HydroShare. Directly connecting to data via the S3 API standard enables us to efficiently upload or download data, as well as perform buffered reading (and lazy data loading) using scientific libraries. In this section we'll demonstrate some of these capabilities.

In order to access HydroShare directly via S3 API, we need to first get an access key and token from HydroShare. The access key and token will be used to establish and authenticate the connection to our data.

In [ ]:
# Query the HydroShare API to gather S3 key and token
response = requests.post(f"https://hydroshare.org/hsapi/user/service/accounts/s3/",
                         auth=(username, password))
key=response.json()['access_key']
secret=response.json()['secret_key']

Now that we have credentials for accessing HydroShare data via its S3 interface, we need to establish a connection using an S3-compatible library that will provide functions that we can use for interacting with the data store. We'll demonstrate how this can be accomplished using the `s3fs` library, although there are alternatives that can also be used.

In [ ]:
import s3fs

In [ ]:
fs = s3fs.S3FileSystem(key=key,
                       secret=secret,
                       endpoint_url=f"https://s3.hydroshare.org")

In [ ]:
# Get the S3 path for accessing the dataset we're interested in
resource_id = '5b379c908019453e86f304e0a07c4984'
res_json = hs._hs_session.get(f"hsapi/resource/s3/{resource_id}/", status_code=200).json()
resource_s3_path = f"{res_json['bucket']}/{res_json['prefix']}"
print(resource_s3_path)

We now have everything we need for interacting with our data. Below are a few common operations that are useful when working with large datasets and large collections of data.

In [ ]:
# Listing/Browsing the content of a HydroShare resource
files = fs.ls(resource_s3_path)
for f in files:
    print(f)

In [ ]:
# List the size of these data
print(f'Total size of files is appoximately: {fs.du(resource_s3_path) / 1e9} GB')

We can download files by simply using the `s3fs` `get` function.

In [ ]:
# Downloading a file
fs.get(f'{resource_s3_path}vpu01_medium_range_troute.parquet', '.')

Similarily, we can upload files using teh `s3fs` `put` function. The code below will only work when you have "edit" permissions on the HydroShare resource. To try this, you'll want to create a new HydroShare resource and replace `test.nc` with a file that you have.

In [ ]:
# Uploading a file
fs.put('f{resource_s3_path}/test.nc', prefix='')

## 3. Interacting with HydroShare Files via Python AWS S3 CLI

This section demonstrates how to interact with HydroShare content using the AWS S3 CLI utility. This is particularly useful when working with large data stored in HydroShare when Python is not available. One example of when this is useful is when moving data around on cloud servers or compute instances. 

The first thing you'll need to do is install the Amazon S3 Command Line Interfance on your computer. Since installation varies by computing platform, please refer to the official documentation for installation instructions: https://docs.aws.amazon.com/cli/latest/userguide/getting-started-install.html.

Once installed, you should see something that resembles the following when running the `aws s3` ccommand in your terminal:

```bash
bash-5.3$ aws s3 help 
S3()                                                                      S3()

NAME
       s3 -

DESCRIPTION
       This section explains prominent concepts and notations in the set of
       high-level S3 commands provided.

       If you are looking for the low level S3 commands for the CLI, please
       see the s3api command reference page.

   Path Argument Type
       Whenever using a command, at least one path argument must be specified.
       There are two types of path arguments: LocalPath and S3Uri.

       LocalPath: represents the path of a local file or directory.  It can be
       written as an absolute path or relative path.

       S3Uri: represents the location of a S3 object, prefix, or bucket.  This
       must be written in the form s3://amzn-s3-demo-bucket/mykey where
       amzn-s3-demo-bucket is the specified S3 bucket, mykey is the specified
       S3 key.  The path argument must begin with s3:// in order to denote
       that the path argument refers to a S3 object. Note that prefixes are
       separated by forward slashes. For example, if the S3 object myobject
       had the prefix myprefix, the S3 key would be myprefix/myobject, and if
       the object was in the bucket amzn-s3-demo-bucket, the S3Uri would be
       s3://amzn-s3-demo-bucket/myprefix/myobject.
       
       ...
       
```

Next, we'll want to create a new "profile" that defines a connection to HydroShare. This can be done by running the following. Note that the name "hydroshare" used in the command below is just a label that we're giving our profile. This will be referenced in future commands, so if you choose to use a alternate name make sure you update subsequent commands to reflect that.

```bash
aws configure --profile hydroshare
```

You'll be prompted for your access key and secret, simply enter the values that were collected in the steps above.

Next, we'll configure the endpoint_url by running the following command:

```bash
aws configure set profile.hydroshare.endpoint_url https://s3.hydroshare.org
```

Now we should be able to query data stored in HydroShare using our credentials!

In [ ]:
!aws s3 ls s3://{resource_s3_path} --profile hydroshare

Now you can use any of the `aws` CLI commands. This means that if you're uploading lots of data you can leverage things like `sync` to move data in an efficient manner ;) 